# BITS Pilani WILP - M.Tech (AIML)
## Machine Learning (Assignment - 2)
### End-to-End Comparative Evaluation of 5 Classification Algorithms on Telco Customer Churn

**Student Name / ID**: M.Tech AIML Student  
**Course**: Machine Learning  
**Dataset**: Telco Customer Churn (IBM / Kaggle)  

---
### Assignment Scope:
1. **Dataset Choice**: Telco Customer Churn ($>500$ instances, $>12$ features, binary classification).
2. **Implement 5 Mandated Classification Models**:
   - Logistic Regression
   - Decision Tree Classifier
   - K-Nearest Neighbors Classifier (KNN)
   - Naive Bayes Classifier (GaussianNB)
   - Support Vector Machine (SVM)
3. **Compute 6 Evaluation Metrics for Each Model**:
   - Accuracy
   - AUC Score (ROC-AUC)
   - Precision
   - Recall
   - F1 Score
   - Matthews Correlation Coefficient (MCC)
4. **Export Artifacts & Test Dataset**:
   - Save `test_data.csv` to project root.
   - Save preprocessor and 5 pipeline models as `.joblib` in `model/`.

### Step 1: Import Necessary Libraries

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_curve
)

print("All libraries imported successfully.")

### Step 2: Load and Clean Dataset
- Load `data/Customer-Churn.csv`
- Inspect data types and missing values
- Clean `TotalCharges` (coerce blank strings to numeric 0.0)
- Drop `customerID` (non-predictive identifier)

In [ ]:
data_path = os.path.join("..", "data", "Customer-Churn.csv")
df = pd.read_csv(data_path)
print(f"Initial shape: {df.shape}")

# Drop customerID
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Clean TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)

print(f"Cleaned shape: {df.shape}")
print("Target class distribution:")
print(df['Churn'].value_counts(normalize=True) * 100)
df.head()

### Step 3: Stratified Train-Test Split (80/20) and Export `test_data.csv`
The assignment rubric requires saving `test_data.csv` to the project root for out-of-sample evaluation in Streamlit.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

# 80-20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Save test_data.csv to project root
test_data = X_test.copy()
test_data['Churn'] = y_test
test_data.to_csv(os.path.join("..", "test_data.csv"), index=False)
print(f"Saved test_data.csv with shape: {test_data.shape}")

### Step 4: Build Preprocessing Pipeline
- Numerical features (`tenure`, `MonthlyCharges`, `TotalCharges`) -> `StandardScaler`
- Categorical features -> `OneHotEncoder(handle_unknown='ignore')`

In [ ]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)

# Fit preprocessor and save
preprocessor.fit(X_train)
joblib.dump(preprocessor, "preprocessor.joblib")
print("Preprocessor saved successfully.")

### Step 5: Define, Train, and Evaluate the 5 Mandated Models
We train each classifier inside a Scikit-Learn `Pipeline` with the fitted preprocessor.

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, kernel='rbf', C=1.0, random_state=42)
}

filenames = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Decision Tree': 'decision_tree.joblib',
    'KNN': 'knn.joblib',
    'Naive Bayes': 'naive_bayes.joblib',
    'Support Vector Machine': 'svm.joblib'
}

y_test_bin = (y_test == 'Yes').astype(int)
results = []

for name, clf in classifiers.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    pipeline.fit(X_train, y_train)
    joblib.dump(pipeline, filenames[name])
    
    y_pred = pipeline.predict(X_test)
    classes = list(pipeline.classes_)
    yes_idx = classes.index('Yes')
    y_proba = pipeline.predict_proba(X_test)[:, yes_idx]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test_bin, y_proba)
    prec = precision_score(y_test, y_pred, pos_label='Yes', zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label='Yes', zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label='Yes', zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

results_df = pd.DataFrame(results)
with open('metrics_summary.json', 'w') as f:
    json.dump(results, f, indent=4)

results_df

### Step 6: Visualizations (Confusion Matrices & ROC Curves)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
for name in classifiers.keys():
    model = joblib.load(filenames[name])
    classes = list(model.classes_)
    yes_idx = classes.index('Yes')
    y_proba = model.predict_proba(X_test)[:, yes_idx]
    fpr, tpr, _ = roc_curve(y_test_bin, y_proba)
    score = roc_auc_score(y_test_bin, y_proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {score:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', label='Chance (0.500)')
axes[0].set_title("ROC Curves Comparison", fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")
axes[0].grid(True, linestyle=":", alpha=0.6)

# Multi-metric bar chart
plot_df = results_df.melt(id_vars=["ML Model Name"], var_name="Metric", value_name="Score")
sns.barplot(data=plot_df, x="Metric", y="Score", hue="ML Model Name", ax=axes[1])
axes[1].set_title("Performance Metrics Across Models", fontweight="bold")
axes[1].set_ylim(0, 1.05)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(axis='y', linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()